In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

st.title("Coğrafi İlan Dağılım Haritası ve Lokasyon Keşfi")
st.caption("Seçilen bölgedeki taşınmazların konumsal yoğunlukları, alanları ve metrekare birim fiyat dağılımları.")

if 'df' not in st.session_state:
    st.warning("Lütfen önce ana sayfayı (app.py) çalıştırınız.")
    st.stop()

df = st.session_state['df']

map_ilce_list = ["Tüm İlçeler"] + sorted(df['İlçe'].unique().tolist())
map_ilce = st.selectbox("Harita İlçe Seçimi (İstanbul):", map_ilce_list, key="map_ilce")

coords = {
    'Ataşehir': (40.9833, 29.1167), 'Kadıköy': (40.9903, 29.0275), 'Üsküdar': (41.0244, 29.0050),
    'Beşiktaş': (41.0422, 29.0067), 'Şişli': (41.0600, 28.9870), 'Bakırköy': (40.9800, 28.8700),
    'Avcılar': (40.9801, 28.7175), 'Esenyurt': (41.0342, 28.6801), 'Pendik': (40.8750, 29.2333)
}

map_filtered = df.copy()
if map_ilce != "Tüm İlçeler": 
    map_filtered = map_filtered[map_filtered['İlçe'] == map_ilce]

if len(map_filtered) == 0:
    st.warning("Seçilen lokasyonda haritada gösterilecek ilan kaydı bulunamadı.")
else:
    def get_coords(row):
        for name, (lat, lon) in coords.items():
            if name.lower() in str(row['İlçe']).lower():
                return pd.Series([lat, lon])
        return pd.Series([41.0082, 28.9784])

    map_filtered[['lat', 'lon']] = map_filtered.apply(get_coords, axis=1)
    np.random.seed(42)
    map_filtered['lat'] += np.random.normal(0, 0.006, len(map_filtered))
    map_filtered['lon'] += np.random.normal(0, 0.006, len(map_filtered))

    fig_map = px.scatter_mapbox(
        map_filtered.head(600),
        lat="lat", lon="lon",
        color="Fiyat", size="m² (Brüt)",
        hover_name="İlan Başlığı",
        hover_data={"İlçe": True, "Mahalle": True, "Fiyat": ":,.0f TL", "m² (Brüt)": True, "lat": False, "lon": False},
        color_continuous_scale="Reds",
        size_max=15, zoom=9.5,
        center={"lat": 41.02, "lon": 28.95},
        mapbox_style="carto-darkmatter",
        height=600
    )
    fig_map.update_layout(margin=dict(l=0, r=0, t=0, b=0), template="plotly_dark")
    st.plotly_chart(fig_map, use_container_width=True)

    st.divider()
    st.markdown("### Harita Üzerindeki Bölgenin İstatistiksel Özeti")
    hm1, hm2, hm3 = st.columns(3)
    hm1.metric("Bölgedeki İlan Sayısı", f"{len(map_filtered):,} Adet")
    hm2.metric("Bölge Ortalama Satış Bedeli", f"{map_filtered['Fiyat'].mean():,.0f} TL")
    hm3.metric("Bölge Ortalama m² Birim Fiyatı", f"{map_filtered['fiyat_m2'].mean():,.0f} TL")